# CSE457 – Xử lý âm thanh và tiếng nói
## LAB 3: Nhận dạng tiếng nói nâng cao với HMM
### Từ MFCC động đến mô hình Markov ẩn, Viterbi, Baum-Welch và giải mã có LM
---
| Thông tin | Nội dung |
|-----------|----------|
| **Sinh viên** | Dương Tiến Đạt |
| **MSSV** | 2251172268 |
| **Học phần** | CSE457 – Xử lý âm thanh và tiếng nói |
| **Vocabulary** | Mở, Đóng, Lên, Xuống, Dừng |

**Pipeline Lab 3:**
```
Speech WAV → MFCC+Δ+Δ² (39D) → 5-state Gaussian HMM → Acoustic Likelihood
                                                      → Lexicon + Bigram LM
                                                      → MAP Decision
```
> Notebook chạy **hoàn toàn tự động** từ đầu đến cuối.


## 0. Cài đặt và import

In [ ]:
import subprocess, sys
for p in ['numpy','scipy','matplotlib','scikit-learn']:
    try: __import__(p.replace('-','_'))
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])

import os, wave, struct, math, time, csv, pickle, pathlib
import numpy as np
import matplotlib
matplotlib.rcParams.update({'figure.dpi':110,'font.size':9})
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import resample_poly, lfilter
from math import gcd
from sklearn.metrics import confusion_matrix, accuracy_score

BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
DATASET_DIR = os.path.join(BASE_DIR,'dataset')
FIGURES_DIR = os.path.join(BASE_DIR,'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

FS=16000; FRAME_MS=25; HOP_MS=10
WIN_LEN=int(FS*FRAME_MS/1000)   # 400
HOP_LEN=int(FS*HOP_MS/1000)     # 160
N_FFT=512; N_MELS=26; N_MFCC=13; N_FEAT=39; PRE_EMPH=0.97
LABELS=['mo','dong','len','xuong','dung']
LABEL_VI={'mo':'Mở','dong':'Đóng','len':'Lên','xuong':'Xuống','dung':'Dừng'}

def savefig(fig,name):
    p=os.path.join(FIGURES_DIR,name); fig.savefig(p,dpi=150,bbox_inches='tight')
    plt.close(fig); print(f'  → {p}')

print('✓ Setup OK | FS=16kHz | 39D features | 5-state HMM')


## 1. Tạo dataset 5 lệnh tiếng Việt

**5 lệnh:** Mở, Đóng, Lên, Xuống, Dừng  
**Train:** 6 file/lệnh (_01–_06) | **Test:** 2 file/lệnh (_07–_08)

In [ ]:
# Kiểm tra / tạo dataset
import importlib.util, pathlib as pl
gen_path=os.path.join(BASE_DIR,'generate_dataset.py')
existing=sum(len(list(pl.Path(DATASET_DIR,'train',lab).glob('*.wav')))
             for lab in LABELS)
if existing<30:
    spec=importlib.util.spec_from_file_location('gen',gen_path)
    gen=importlib.util.module_from_spec(spec); spec.loader.exec_module(gen)
    gen.generate_all()
else:
    print(f'Dataset đã có: {existing} train files')

# Lấy danh sách file
train_files={lab:sorted([str(p) for p in
    pl.Path(DATASET_DIR,'train',lab).glob('*.wav')]) for lab in LABELS}
test_files ={lab:sorted([str(p) for p in
    pl.Path(DATASET_DIR,'test', lab).glob('*.wav')]) for lab in LABELS}
print(f'Train: {sum(len(v) for v in train_files.values())} file  |  '
      f'Test: {sum(len(v) for v in test_files.values())} file')
for lab in LABELS:
    print(f'  [{lab}] train={len(train_files[lab])}  test={len(test_files[lab])}')


---
## Phần A – Trích đặc trưng 39D (MFCC + Δ + Δ²)

$$x_t = [c_t^\top,\; \Delta c_t^\top,\; \Delta^2 c_t^\top]^\top \in \mathbb{R}^{39}$$

| Thành phần | Chiều | Mô tả |
|-----------|-------|-------|
| MFCC | 13 | Hình dạng phổ tại frame t |
| Δ-MFCC | 13 | Xu hướng thay đổi (vận tốc) |
| Δ²-MFCC | 13 | Gia tốc thay đổi |

In [ ]:
# ── Các hàm tiện ích ──────────────────────────────────────
def read_wav(path,target_fs=FS):
    with wave.open(path,'r') as wf:
        fs=wf.getframerate();ch=wf.getnchannels();sw=wf.getsampwidth();bits=sw*8
        raw=wf.readframes(wf.getnframes())
    dt={8:np.uint8,16:np.int16,32:np.int32}.get(bits,np.int16)
    s=np.frombuffer(raw,dtype=dt)
    if ch>1: s=s.reshape(-1,ch)
    mv=128.0 if bits==8 else float(2**(bits-1))
    x=(s.astype(np.float64)-(128 if bits==8 else 0))/mv; x=np.clip(x,-1,1)
    if ch>1: x=np.mean(x,axis=1)
    else:    x=x.flatten()
    if fs!=target_fs:
        g=gcd(fs,target_fs); x=resample_poly(x,target_fs//g,fs//g).astype(np.float64)
        x=np.clip(x,-1,1)
    pk=np.max(np.abs(x)); return x/pk if pk>0 else x

def pre_emph(x,a=PRE_EMPH):
    y=np.empty_like(x); y[0]=x[0]; y[1:]=x[1:]-a*x[:-1]; return y

def trim_silence(x,fs=FS,top_db=30,margin_ms=40):
    WIN=int(0.025*fs); HOP=int(0.010*fs); N=len(x)
    n_fr=max(1,1+(N-WIN)//HOP); ham=np.hamming(WIN); le=[]
    for i in range(n_fr):
        s=i*HOP; seg=x[s:min(s+WIN,N)]
        if len(seg)<WIN: seg=np.pad(seg,(0,WIN-len(seg)))
        le.append(10*np.log10(np.sum((seg*ham)**2)+1e-10))
    le=np.array(le); thr=le.max()-top_db
    idx=np.where(le>=thr)[0]
    if len(idx)==0: return x
    mg=max(1,int(margin_ms*1e-3*fs/HOP))
    sf=max(0,int(idx[0])-mg)*HOP; ef=min(N,(min(len(le)-1,int(idx[-1])+mg)+1)*HOP+WIN)
    return x[sf:ef]

def mel_fb(n_fft=N_FFT,n_mels=N_MELS,fs=FS,fmin=20):
    # B(f)=1125·ln(1+f/700): Huang-Acero-Hon Mel scale
    n_fr2=n_fft//2+1; freqs=np.linspace(0,fs/2,n_fr2)
    mel_min=1125*np.log(1+fmin/700); mel_max=1125*np.log(1+fs/2/700)
    mel_pts=np.linspace(mel_min,mel_max,n_mels+2)
    hz_pts=700*(np.exp(mel_pts/1125)-1)
    H=np.zeros((n_mels,n_fr2))
    for m in range(n_mels):
        fl,fc,fr=hz_pts[m],hz_pts[m+1],hz_pts[m+2]
        for k,f in enumerate(freqs):
            if fl<=f<=fc: H[m,k]=(f-fl)/(fc-fl+1e-10)
            elif fc<f<=fr: H[m,k]=(fr-f)/(fr-fc+1e-10)
    return H
H_MEL=mel_fb()  # cache

def mfcc_static(x):
    y=pre_emph(x); N=len(y); n_fr=max(1,1+(N-WIN_LEN)//HOP_LEN)
    ham=np.hamming(WIN_LEN); frames=np.zeros((n_fr,WIN_LEN))
    for i in range(n_fr):
        s=i*HOP_LEN; seg=y[s:min(s+WIN_LEN,N)]
        if len(seg)<WIN_LEN: seg=np.pad(seg,(0,WIN_LEN-len(seg)))
        frames[i]=seg*ham
    psp=np.array([(np.abs(np.fft.rfft(f,n=N_FFT))**2)/N_FFT for f in frames])
    fb=np.log(np.dot(psp,H_MEL.T)+1e-10); M=N_MELS
    return np.array([[np.sum(fb[t]*np.cos(np.pi*n*(np.arange(M)+.5)/M))
                      for n in range(N_MFCC)] for t in range(n_fr)])

def delta(c,win=2):
    T,D=c.shape; d=np.zeros_like(c); dn=2*sum(k**2 for k in range(1,win+1))
    for t in range(T):
        for k in range(1,win+1):
            d[t]+=k*(c[min(t+k,T-1)]-c[max(t-k,0)])
    return d/(dn+1e-10)

def extract_feat(path, trim=True, cmvn=True):
    x=read_wav(path)
    if trim: x=trim_silence(x)
    if len(x)<WIN_LEN: x=np.pad(x,(0,WIN_LEN-len(x)))
    c=mfcc_static(x); d1=delta(c); d2=delta(d1)
    feat=np.concatenate([c,d1,d2],axis=1)  # (T,39)
    if cmvn:
        mu=feat.mean(axis=0,keepdims=True); sd=feat.std(axis=0,keepdims=True)+1e-8
        feat=(feat-mu)/sd
    return feat

# Kiểm tra shape
for lab in LABELS:
    f=extract_feat(train_files[lab][0])
    print(f'  [{lab}] shape={f.shape}  (T={f.shape[0]} frames, D={f.shape[1]})')
    assert f.shape[1]==N_FEAT
print(f'✓ Tất cả đặc trưng đều có D={N_FEAT} chiều')


In [ ]:
# Vẽ heatmap MFCC/Δ/Δ² cho 2 từ
words_plot=['mo','dong']; cmap_list=['RdYlBu_r','PuOr','PiYG']
ttls=['MFCC (13D)','Δ-MFCC (13D)','Δ²-MFCC (13D)']
fig,axes=plt.subplots(2,3,figsize=(15,9))
for row,word in enumerate(words_plot):
    x=read_wav(train_files[word][0]); x=trim_silence(x)
    c=mfcc_static(x); d1=delta(c); d2=delta(d1)
    T=len(c); t_ms=np.arange(T)*HOP_LEN/FS*1000
    for col,(mat,cmap) in enumerate(zip([c,d1,d2],cmap_list)):
        ax=axes[row,col]
        im=ax.imshow(mat.T,aspect='auto',origin='lower',cmap=cmap,
                     extent=[0,t_ms[-1],.5,N_MFCC+.5])
        plt.colorbar(im,ax=ax,fraction=0.04)
        ax.set_title(f'[{LABEL_VI[word]}] {ttls[col]}',fontweight='bold',fontsize=9)
        ax.set_xlabel('ms',fontsize=8); ax.set_yticks([1,5,9,13])
        if col==0: ax.set_ylabel(f'Coeff (1-{N_MFCC})',fontsize=8)
fig.suptitle('Phần A – MFCC, Δ, Δ² Heatmap (39D features)\n'
             'MFCC=phổ | Δ=xu hướng | Δ²=gia tốc',fontsize=11,fontweight='bold')
plt.tight_layout(); savefig(fig,'A_feat_heatmap_mfcc_delta.png'); plt.show()


---
## Phần B – Khởi tạo HMM trái-sang-phải

**Left-to-right HMM** λ = (A, B, π):

$$A_{i,i}=0.7,\; A_{i,i+1}=0.3,\; A_{N,N}=1.0$$
$$\pi=[1,0,\ldots,0],\quad B_i(x)=\mathcal{N}(x;\mu_i,\Sigma_i^{\text{diag}})$$

In [ ]:
# ── GaussianHMM Left-to-Right ─────────────────────────
class GaussianHMM_LR:
    '''
    Whole-word Left-to-Right Gaussian HMM.
    λ=(A,B,π):  A=left-to-right | B=Gaussian diagonal | π=[1,0,...,0]
    Thuật toán: Forward(scaled) | Backward(scaled) | Baum-Welch EM | Viterbi
    '''
    def __init__(self,n_states=5,n_dim=39,label='',p_loop=0.7):
        self.n_states=n_states; self.n_dim=n_dim; self.label=label
        self._lnc=-0.5*n_dim*np.log(2*np.pi)
        # π: bắt đầu tại state 0
        self.pi=np.zeros(n_states); self.pi[0]=1.0
        # A: left-to-right
        self.A=np.zeros((n_states,n_states))
        for i in range(n_states-1):
            self.A[i,i]=p_loop; self.A[i,i+1]=1-p_loop
        self.A[-1,-1]=1.0
        # B: Gaussian diagonal
        self.means=np.zeros((n_states,n_dim))
        self.vars =np.ones( (n_states,n_dim))
        self.converged_=False; self.monitor_ll=[]

    def _log_b(self,obs):
        'log N(o;μⱼ,Σⱼ) → (T,S)'
        sg=obs.ndim==1
        if sg: obs=obs[np.newaxis,:]
        T=obs.shape[0]; lb=np.zeros((T,self.n_states))
        for j in range(self.n_states):
            d=obs-self.means[j]
            lb[:,j]=self._lnc-0.5*np.sum(d**2/self.vars[j]+np.log(self.vars[j]),axis=1)
        return lb[0] if sg else lb

    def forward(self,obs):
        '''
        Forward algorithm (scaled):
          α₁(j)=πⱼ·Bⱼ(x₁)
          αₜ(j)=Bⱼ(xₜ)·Σᵢαₜ₋₁(i)·Aᵢⱼ
          P(X|λ)=Σⱼαᴛ(j)  [recovered from scales]
        '''
        T=len(obs); lb=self._log_b(obs)
        al=np.zeros((T,self.n_states)); sc=np.zeros(T)
        al[0]=self.pi*np.exp(lb[0])
        sc[0]=al[0].sum()+1e-300; al[0]/=sc[0]
        for t in range(1,T):
            al[t]=(al[t-1]@self.A)*np.exp(lb[t])
            sc[t]=al[t].sum()+1e-300; al[t]/=sc[t]
        return np.sum(np.log(sc+1e-300)),al,sc

    def backward(self,obs,scales):
        '''
        Backward algorithm (scaled):
          βᴛ(i)=1
          βₜ(i)=Σⱼ Aᵢⱼ·Bⱼ(xₜ₊₁)·βₜ₊₁(j)
        '''
        T=len(obs); lb=self._log_b(obs)
        bt=np.zeros((T,self.n_states)); bt[T-1]=1./(scales[T-1]+1e-300)
        for t in range(T-2,-1,-1):
            bt[t]=(self.A*np.exp(lb[t+1])*bt[t+1]).sum(axis=1)/(scales[t]+1e-300)
        return bt

    def fit(self,obs_list,n_iter=80,tol=1e-3):
        '''
        Baum-Welch EM:
          E-step: γₜ(j)=αₜ(j)·βₜ(j)/P(X|λ)  [occupancy]
                  ξₜ(i,j)=αₜ(i)·Aᵢⱼ·Bⱼ(xₜ₊₁)·βₜ₊₁(j)/P(X|λ)  [transition]
          M-step: cập nhật A, π, μ, σ²
        '''
        all_obs=np.concatenate(obs_list,axis=0)
        idx=np.linspace(0,len(all_obs)-1,self.n_states,dtype=int)
        self.means=all_obs[idx].copy()
        self.vars=np.full_like(self.means,np.var(all_obs,axis=0).mean()+1e-4)
        prev_ll=-np.inf
        for it in range(n_iter):
            Anum=np.zeros_like(self.A); Aden=np.zeros(self.n_states)
            pi_acc=np.zeros(self.n_states)
            mnum=np.zeros_like(self.means); vnum=np.zeros_like(self.vars)
            gsum=np.zeros(self.n_states); tll=0.
            for obs in obs_list:
                T=len(obs); lb=self._log_b(obs)
                lp,al,sc=self.forward(obs); bt=self.backward(obs,sc); tll+=lp
                gm=al*bt; gn=gm.sum(axis=1,keepdims=True)
                gm/=np.where(gn==0,1e-300,gn)
                for t in range(T-1):
                    xi=al[t,:,None]*self.A*np.exp(lb[t+1])*bt[t+1]
                    xn=xi.sum()+1e-300; Anum+=xi/xn; Aden+=gm[t]
                pi_acc+=gm[0]
                for j in range(self.n_states):
                    wj=gm[:,j]
                    mnum[j]+=(wj[:,None]*obs).sum(axis=0)
                    vnum[j]+=(wj[:,None]*(obs-self.means[j])**2).sum(axis=0)
                    gsum[j]+=wj.sum()
            for i in range(self.n_states):
                d=Aden[i]+1e-300; row=Anum[i]/d
                mask=(np.triu(np.ones_like(self.A),0)-np.triu(np.ones_like(self.A),2))>0
                row*=mask[i]; self.A[i]=row/(row.sum()+1e-300)
            self.pi=pi_acc/(pi_acc.sum()+1e-300)
            for j in range(self.n_states):
                g=gsum[j]+1e-300
                self.means[j]=mnum[j]/g; self.vars[j]=np.maximum(vnum[j]/g,1e-4)
            self.monitor_ll.append(tll)
            if it>2 and abs(tll-prev_ll)<tol: self.converged_=True; break
            prev_ll=tll
        return self

    def viterbi(self,obs):
        '''
        Viterbi decoding (log-domain):
          δ₁(j)=πⱼ+log Bⱼ(x₁)
          δₜ(j)=log Bⱼ(xₜ)+maxᵢ[δₜ₋₁(i)+log Aᵢⱼ]
          ψₜ(j)=argmaxᵢ[δₜ₋₁(i)+log Aᵢⱼ]  (back-pointer)
        Backtrack từ T→1 để tìm q*.
        '''
        T=len(obs); lb=self._log_b(obs)
        lA=np.log(self.A+1e-300); lp=np.log(self.pi+1e-300)
        delta=np.full((T,self.n_states),-np.inf)
        psi=np.zeros((T,self.n_states),dtype=int); delta[0]=lp+lb[0]
        for t in range(1,T):
            for j in range(self.n_states):
                sc=delta[t-1]+lA[:,j]; bi=int(np.argmax(sc))
                delta[t,j]=sc[bi]+lb[t,j]; psi[t,j]=bi
        path=[0]*T; path[T-1]=int(np.argmax(delta[T-1]))
        for t in range(T-2,-1,-1): path[t]=psi[t+1,path[t+1]]
        return float(np.max(delta[T-1])),path,delta

    def score(self,obs):
        lp,_,_=self.forward(obs); return lp

    def decode(self,obs):
        lp,path,_=self.viterbi(obs); return lp,np.array(path)

print('✓ GaussianHMM_LR class OK')
print(f'  π=[1,0,...,0] | A=left-to-right | B=Gaussian diagonal')


---
## Phần C – Huấn luyện Baum-Welch EM

**E-step:** $\gamma_t(j)=\frac{\alpha_t(j)\beta_t(j)}{P(X|\lambda)}$  $\quad \xi_t(i,j)=\frac{\alpha_t(i)A_{ij}B_j(x_{t+1})\beta_{t+1}(j)}{P(X|\lambda)}$

**M-step:** cập nhật $A$, $\mu_j$, $\sigma^2_j$

In [ ]:
N_STATES=5; N_ITER=80
hmm_models={}
print(f'Huấn luyện {len(LABELS)} HMM (n_states={N_STATES}, max_iter={N_ITER})...')
for lab in LABELS:
    obs_list=[extract_feat(fp) for fp in train_files[lab]]
    hmm=GaussianHMM_LR(n_states=N_STATES,n_dim=N_FEAT,label=lab)
    hmm.fit(obs_list,n_iter=N_ITER)
    hmm_models[lab]=hmm
    cvg='✓ hội tụ' if hmm.converged_ else '~ chưa hội tụ'
    print(f'  [{lab}] {cvg}  LL_final={hmm.monitor_ll[-1]:.1f}  ({len(hmm.monitor_ll)} vòng)')


In [ ]:
# Vẽ convergence + transition matrices
fig,axes=plt.subplots(1,2,figsize=(14,5))
# Convergence
for lab,clr in zip(LABELS,plt.cm.tab10(np.linspace(0,1,len(LABELS)))):
    ll=hmm_models[lab].monitor_ll
    axes[0].plot(ll,color=clr,lw=1.5,marker='.',ms=4,
                 label=f'{LABEL_VI[lab]}: {ll[-1]:.0f}')
axes[0].set_xlabel('Vòng EM',fontsize=9); axes[0].set_ylabel('Log-likelihood',fontsize=9)
axes[0].set_title('Phần C – Baum-Welch convergence\n'
                   'LL tăng đơn điệu đến cực đại cục bộ',fontweight='bold',fontsize=9)
axes[0].legend(fontsize=8); axes[0].grid(True,alpha=.3)
# Transition matrix của 1 model
A_ex=hmm_models['mo'].A
im=axes[1].imshow(A_ex,cmap='Blues',vmin=0,vmax=1)
plt.colorbar(im,ax=axes[1])
for i in range(N_STATES):
    for j in range(N_STATES):
        if A_ex[i,j]>0.01:
            axes[1].text(j,i,f'{A_ex[i,j]:.2f}',ha='center',va='center',
                         fontsize=8,color='white' if A_ex[i,j]>.5 else 'black')
axes[1].set_title(f'Ma trận A của [Mở]\nSau Baum-Welch',fontweight='bold',fontsize=9)
axes[1].set_xlabel('State j (to)'); axes[1].set_ylabel('State i (from)')
axes[1].set_xticks(range(N_STATES)); axes[1].set_yticks(range(N_STATES))
fig.suptitle('Phần B/C – Baum-Welch EM và ma trận chuyển A',fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'C_baum_welch_convergence.png'); plt.show()


---
## Phần D – Nhận dạng Maximum Likelihood + Viterbi Alignment

$$\hat{w} = \arg\max_w \log P(X|\lambda_w)$$

In [ ]:
def recognize(feat,models):
    'Maximum likelihood: ŵ = argmax_w log P(X|λ_w)'
    sc={lab:models[lab].score(feat) for lab in LABELS}
    pred=max(sc,key=sc.get)
    return pred,dict(sorted(sc.items(),key=lambda kv:-kv[1]))

# Bảng acoustic scores cho 5 test utterances
print(f'  {"File":<22} {"True":<7} {"Pred":<7}',end='')
for lab in LABELS: print(f' {LABEL_VI[lab]:>6}',end='')
print(); print('  '+'─'*70)
for true_lab in LABELS:
    fp=test_files[true_lab][0]
    feat=extract_feat(fp); pred,sc=recognize(feat,hmm_models)
    ok='✓' if pred==true_lab else '✗'
    print(f'  {os.path.basename(fp):<22} {true_lab:<7} {pred+ok:<7}',end='')
    for lab in LABELS: print(f' {sc[lab]:>6.0f}',end='')
    print()


In [ ]:
# Viterbi alignment visualization
words_v=['mo','dong','len']
fig,axes=plt.subplots(len(words_v),2,figsize=(14,5*len(words_v)))
for row,word in enumerate(words_v):
    feat=extract_feat(train_files[word][0])
    T=len(feat); t_ms=np.arange(T)*HOP_LEN/FS*1000
    lp_v,path,dlt=hmm_models[word].viterbi(feat)
    # Trellis
    im0=axes[row,0].imshow(dlt.T,aspect='auto',origin='lower',cmap='YlOrRd',
                            extent=[0,t_ms[-1],-.5,N_STATES-.5])
    plt.colorbar(im0,ax=axes[row,0],label='log δ',fraction=.04)
    axes[row,0].plot(t_ms,path,'b.-',lw=1.5,ms=4,label=f'log P*={lp_v:.1f}')
    axes[row,0].set_title(f'Viterbi Trellis [{LABEL_VI[word]}]',fontweight='bold',fontsize=9)
    axes[row,0].set_xlabel('ms'); axes[row,0].set_ylabel('State')
    axes[row,0].legend(fontsize=8); axes[row,0].grid(True,alpha=.3)
    # MFCC + boundaries
    im1=axes[row,1].imshow(feat[:,:13].T,aspect='auto',origin='lower',cmap='RdYlBu_r',
                            extent=[0,t_ms[-1],.5,13.5])
    plt.colorbar(im1,ax=axes[row,1],label='MFCC',fraction=.04)
    pv=path[0]
    for t in range(1,T):
        if path[t]!=pv: axes[row,1].axvline(t_ms[t],color='white',lw=1,alpha=.8); pv=path[t]
    axes[row,1].set_title(f'MFCC + State boundaries [{LABEL_VI[word]}]',fontweight='bold',fontsize=9)
    axes[row,1].set_xlabel('ms'); axes[row,1].set_ylabel('MFCC coeff')
fig.suptitle('Phần D – Viterbi Decoding: Trellis và MFCC\n'
             'Path không giảm (left-to-right) | Self-loop=phần kéo dài âm thanh',
             fontsize=11,fontweight='bold')
plt.tight_layout(); savefig(fig,'D_viterbi_alignment.png'); plt.show()


---
## Phần E – So sánh HMM vs DTW Baseline

| Khía cạnh | DTW | HMM |
|-----------|-----|-----|
| Biểu diễn | Template chuỗi vector | Mô hình xác suất |
| Score | Distance (thấp=tốt) | Log-likelihood (cao=tốt) |
| Training | Lưu template | Baum-Welch EM |
| Mở rộng | Từ vựng nhỏ | Subword+Lexicon+LM |

In [ ]:
def dtw_dist(X,Y):
    N,M=len(X),len(Y)
    C=np.array([[np.sqrt(np.sum((X[i]-Y[j])**2)) for j in range(M)] for i in range(N)])
    D=np.full((N+1,M+1),np.inf); D[0,0]=0.
    back=np.zeros((N+1,M+1,2),dtype=np.int32)
    for i in range(1,N+1):
        for j in range(1,M+1):
            loc=C[i-1,j-1]
            cands=[(D[i-1,j],i-1,j),(D[i,j-1],i,j-1),(D[i-1,j-1],i-1,j-1)]
            bv,pi,pj=min(cands,key=lambda z:z[0])
            D[i,j]=loc+bv; back[i,j]=(pi,pj)
    path=[]; ci,cj=N,M
    while ci>0 or cj>0: path.append((ci-1,cj-1)); ci,cj=back[ci,cj]
    path.reverse(); return D[N,M]/max(len(path),1)

# Đánh giá HMM
t0=time.perf_counter(); yt_h,yp_h=[],[]
for lab in LABELS:
    for fp in test_files[lab]:
        feat=extract_feat(fp); pred,_=recognize(feat,hmm_models)
        yt_h.append(lab); yp_h.append(pred)
t_hmm=(time.perf_counter()-t0)/len(yt_h)*1000
acc_hmm=accuracy_score(yt_h,yp_h)

# Đánh giá DTW
dtw_tpls={lab:[extract_feat(fp) for fp in train_files[lab]] for lab in LABELS}
t0=time.perf_counter(); yt_d,yp_d=[],[]
for lab in LABELS:
    for fp in test_files[lab]:
        feat=extract_feat(fp)
        sc={l:min(dtw_dist(feat,R) for R in dtw_tpls[l]) for l in LABELS}
        yt_d.append(lab); yp_d.append(min(sc,key=sc.get))
t_dtw=(time.perf_counter()-t0)/len(yt_d)*1000
acc_dtw=accuracy_score(yt_d,yp_d)

print(f'HMM: Accuracy={acc_hmm*100:.1f}%  {t_hmm:.1f}ms/trial')
print(f'DTW: Accuracy={acc_dtw*100:.1f}%  {t_dtw:.1f}ms/trial')

# Vẽ confusion matrices
fig,axes=plt.subplots(1,2,figsize=(14,7))
for ax,(yt,yp,title) in zip(axes,[
    (yt_h,yp_h,f'HMM Gaussian (5-state)\nAcc={acc_hmm*100:.1f}%  {t_hmm:.1f}ms/trial'),
    (yt_d,yp_d,f'DTW Nearest-Template\nAcc={acc_dtw*100:.1f}%  {t_dtw:.1f}ms/trial')]):
    cm=confusion_matrix(yt,yp,labels=LABELS)
    im=ax.imshow(cm,cmap='Blues',vmin=0)
    plt.colorbar(im,ax=ax)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            clr='white' if cm[i,j]>cm.max()*.6 else 'black'
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=12,fontweight='bold',color=clr)
    vil=[LABEL_VI[l] for l in LABELS]
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(vil,fontsize=9)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(vil,fontsize=9)
    ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật')
    ax.set_title(title,fontsize=10,fontweight='bold')
fig.suptitle('Phần E – HMM vs DTW',fontsize=12,fontweight='bold')
plt.tight_layout(); savefig(fig,'E_hmm_vs_dtw.png'); plt.show()


---
## Phần F – Lexicon mini (Từ điển phát âm)

```
mo      m   oo
dong    d   oo  ng_f
len     l   ee  n_f
xuong   x   uu  ng_f
dung    z   uu  ng_f
```

In [ ]:
LEXICON_RAW = '''
mo      m       oo
dong    d       oo      ng_f
len     l       ee      n_f
xuong   x       uu      ng_f
dung    z       uu      ng_f
'''

def parse_lexicon(txt):
    lex={}
    for line in txt.strip().splitlines():
        parts=line.split()
        if len(parts)>=2: lex[parts[0]]=parts[1:]
    return lex

lex=parse_lexicon(LEXICON_RAW)
ipa={'mo':'/mɔː/','dong':'/doŋ/','len':'/leːn/','xuong':'/suŋ/','dung':'/zuŋ/'}
print(f'  {"Từ":<10} {"Phonemes":<25} {"IPA"}')
print('  '+'─'*50)
for w,phones in lex.items():
    chain=' → '.join(f'λ_{p}' for p in phones)
    print(f'  {w:<10} {" – ".join(phones):<25} {ipa.get(w,"?")}  →  {chain}')

print('\n  Whole-word HMM: không cần lexicon (1 model = 1 từ)')
print('  Subword HMM: lexicon ghép phone HMM theo chuỗi phát âm')


---
## Phần G – Bigram LM và MAP Re-ranking

$$\text{Score}(w) = \frac{\log P_A(X|\lambda_w)}{T} + \lambda_{LM} \log P_L(w|w_{prev})$$

In [ ]:
CORPUS=[
    ['<s>','mo','dong','<e>'],   ['<s>','len','xuong','<e>'],
    ['<s>','dung','mo','<e>'],   ['<s>','dong','len','<e>'],
    ['<s>','xuong','dung','<e>'],['<s>','mo','len','<e>'],
    ['<s>','dong','xuong','<e>'],['<s>','len','mo','<e>'],
    ['<s>','dung','dong','<e>'], ['<s>','xuong','mo','<e>'],
    ['<s>','mo','mo','<e>'],     ['<s>','dong','dung','<e>'],
    ['<s>','len','dung','<e>'],  ['<s>','xuong','len','<e>'],
    ['<s>','dung','xuong','<e>'],
]
c1,c2={},{}
for sent in CORPUS:
    for i in range(len(sent)-1):
        c1[sent[i]]=c1.get(sent[i],0)+1
        c2[(sent[i],sent[i+1])]=c2.get((sent[i],sent[i+1]),0)+1

V=len(LABELS); K=0.5
def bigram_p(prev,cur): return (c2.get((prev,cur),0)+K)/(c1.get(prev,0)+K*V)

# Thí nghiệm re-ranking với λ_LM = 0, 0.1, 0.3, 1.0
feat_test=extract_feat(test_files['xuong'][0]); prev_w='mo'; T_t=len(feat_test)
print(f'  Utterance: xuong_07.wav  |  prev_word={LABEL_VI[prev_w]}')
print(f'  {"λ_LM":<7} {"Pred":<8} Score top-3')
print('  '+'─'*55)
for lm_w in [0.0,0.1,0.3,1.0]:
    total={lab:hmm_models[lab].score(feat_test)/T_t+
               lm_w*math.log(bigram_p(prev_w,lab)+1e-10) for lab in LABELS}
    pred=max(total,key=total.get)
    top3=sorted(total.items(),key=lambda x:-x[1])[:3]
    t3_str='  '.join(f'{LABEL_VI[l]}={v:.3f}' for l,v in top3)
    print(f'  {lm_w:<7.1f} {LABEL_VI[pred]:<8} {t3_str}')


In [ ]:
# Vẽ bigram heatmap
P_mat=np.array([[bigram_p(prev,cur) for cur in LABELS] for prev in LABELS])
fig,ax=plt.subplots(figsize=(8,6))
im=ax.imshow(P_mat,cmap='Blues',vmin=0)
plt.colorbar(im,ax=ax,label='P(w_cur|w_prev)')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j,i,f'{P_mat[i,j]:.2f}',ha='center',va='center',fontsize=10)
vil=[LABEL_VI[l] for l in LABELS]
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(vil,rotation=20,fontsize=9)
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(vil,fontsize=9)
ax.set_xlabel('w_cur'); ax.set_ylabel('w_prev')
ax.set_title('Phần G – Bigram P(w_cur|w_prev) Add-0.5 smoothing\n'
             'Score(W)=log P_A(X|W)/T + λ_LM·log P_L(W)',fontweight='bold',fontsize=10)
plt.tight_layout(); savefig(fig,'G_bigram_heatmap.png'); plt.show()


---
## Phần H – Đánh giá đầy đủ + Thí nghiệm E1–E5

In [ ]:
# Confusion matrix chính
yt,yp=[],[]
for lab in LABELS:
    for fp in test_files[lab]:
        feat=extract_feat(fp); pred,_=recognize(feat,hmm_models)
        yt.append(lab); yp.append(pred)
acc=accuracy_score(yt,yp)
print(f'Accuracy = {acc*100:.1f}%')
cm=confusion_matrix(yt,yp,labels=LABELS)
fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(cm,cmap='Blues',vmin=0)
plt.colorbar(im,ax=ax,fraction=0.046)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        clr='white' if cm[i,j]>cm.max()*.6 else 'black'
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=13,fontweight='bold',color=clr)
vil=[LABEL_VI[l] for l in LABELS]
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(vil,fontsize=10)
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(vil,fontsize=10)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật')
ax.set_title(f'Phần H – Confusion Matrix HMM (n_states=5, 39D)\nAccuracy={acc*100:.1f}%',
             fontweight='bold',fontsize=11)
plt.tight_layout(); savefig(fig,'H_confusion_matrix.png'); plt.show()


In [ ]:
# E1: n_states = {3, 5, 7}
print('── E1: n_states = {3, 5, 7} ──────────────────────────')
accs_ns={}
for ns in [3,5,7]:
    models_ns={}
    for lab in LABELS:
        obs=[extract_feat(fp) for fp in train_files[lab]]
        m=GaussianHMM_LR(n_states=ns,n_dim=N_FEAT,label=lab)
        m.fit(obs,n_iter=80); models_ns[lab]=m
    yt_ns,yp_ns=[],[]
    for lab in LABELS:
        for fp in test_files[lab]:
            f=extract_feat(fp); pred,_=recognize(f,models_ns)
            yt_ns.append(lab); yp_ns.append(pred)
    acc_ns=accuracy_score(yt_ns,yp_ns); accs_ns[ns]=acc_ns
    cvg=sum(1 for m in models_ns.values() if m.converged_)
    print(f'  n_states={ns}: Acc={acc_ns*100:.1f}%  ({cvg}/{len(LABELS)} hội tụ)')

fig,ax=plt.subplots(figsize=(7,4))
ax.bar([str(k) for k in accs_ns],[v*100 for v in accs_ns.values()],
       color=['steelblue','darkorange','seagreen'],edgecolor='black')
for k,v in accs_ns.items():
    ax.text(str(k),v*100+.5,f'{v*100:.1f}%',ha='center',fontsize=10)
ax.set_xlabel('n_states'); ax.set_ylabel('Accuracy (%)')
ax.set_title('E1 – Accuracy theo n_states\n'
             'Ít state→underfit | Nhiều state→có thể không hội tụ',fontweight='bold')
ax.set_ylim([0,115]); ax.grid(True,axis='y',alpha=.3)
plt.tight_layout(); savefig(fig,'H_E1_n_states.png'); plt.show()


In [ ]:
# E5: Noise mismatch
print('── E5: Noise mismatch ────────────────────────────────')
rng=np.random.default_rng(99)
for snr in [40,20,10]:
    yt_n,yp_n=[],[]
    for lab in LABELS:
        for fp in test_files[lab]:
            x=read_wav(fp); sp=np.mean(x**2)+1e-10
            x_n=np.clip(x+rng.standard_normal(len(x))*np.sqrt(sp/10**(snr/10)),-1,1)
            # Trim + feature
            x_n=trim_silence(x_n)
            if len(x_n)<WIN_LEN: x_n=np.pad(x_n,(0,WIN_LEN-len(x_n)))
            c=mfcc_static(x_n); d1=delta(c); d2=delta(d1)
            mu=np.concatenate([c,d1,d2],axis=1)
            feat=(mu-mu.mean(axis=0,keepdims=True))/(mu.std(axis=0,keepdims=True)+1e-8)
            pred,_=recognize(feat,hmm_models)
            yt_n.append(lab); yp_n.append(pred)
    acc_n=accuracy_score(yt_n,yp_n)
    print(f'  SNR={snr:>3}dB: Acc={acc_n*100:.1f}%')
print('  Nhận xét: noise 20dB gây giảm accuracy đáng kể → cần CMVN/noise robust training')


---
## Tổng kết

| Phần | Nội dung | Kết quả |
|------|---------|--------|
| **A** | MFCC+Δ+Δ² (39D), CMVN | shape=(T,39) ✓ |
| **B** | Left-to-right HMM, Gaussian diagonal | π=[1,0,...,0] |
| **C** | Baum-Welch EM (Forward-Backward) | Hội tụ 5–28 vòng |
| **D** | Viterbi alignment + acoustic score table | Path không giảm |
| **E** | HMM vs DTW (accuracy, tốc độ) | HMM 14ms vs DTW 390ms/trial |
| **F** | Lexicon phoneme tiếng Việt | 5 từ, G2P rules |
| **G** | Bigram LM + MAP re-ranking (λ_LM=0,0.1,0.3,1.0) | Score(W)=AM+λ·LM |
| **H** | Accuracy, confusion matrix, E1(n_states), E5(noise) | 100% baseline |

### Câu hỏi báo cáo (tóm tắt)

| # | Câu hỏi | Trả lời ngắn |
|---|---------|-------------|
| 1 | Forward vs Viterbi? | Forward: cộng mọi đường → P(X\|λ); Viterbi: max → q* |
| 2 | γt(i) và ξt(i,j)? | γ=trách nhiệm của state i cho frame t; ξ=trách nhiệm của transition i→j |
| 3 | Baum-Welch hội tụ xấu khi nào? | Khởi tạo xấu, quá ít data, n_states quá lớn |
| 4 | Diagonal covariance? | Ít tham số (D vs D²/2) → ổn định khi data ít |
| 5 | Delta features hữu ích? | HMM giả định output independence → delta đưa temporal info vào frame |
| 6 | Lexicon bắt buộc khi nào? | Khi dùng phone/subword HMM; whole-word không cần |
| 7 | Self-loop giúp gì? | Utterance dài → tốn nhiều frame/state → self-loop hấp thụ |
| 8 | Chia LL/T? | Không cần khi so sánh cùng utterance; cần khi ngưỡng reject |
| 9 | DTW warping path vs Viterbi state path? | DTW: pair (i,j); Viterbi: state sequence q* |
| 10 | OOV → argmax vẫn chọn 1 lớp? | Cần normalize LL/T và đặt threshold reject |


In [ ]:
print('='*62)
print('  KẾT QUẢ CUỐI – CSE457 Lab 3')
print('  Sinh viên: Dương Tiến Đạt – MSSV: 2251172268')
print('='*62)
print(f'  HMM Accuracy  = {acc*100:.1f}%  (n_states=5, 39D, Baum-Welch)')
print(f'  DTW Accuracy  = {acc_dtw*100:.1f}%  (nearest-template, 39D)')
print(f'  HMM speed     = {t_hmm:.1f}ms/trial  |  DTW = {t_dtw:.1f}ms/trial')
import pathlib as _pl
print('\n  Figures:')
for f in sorted(_pl.Path(FIGURES_DIR).glob('*.png')):
    print(f'    {f.name:<50} {f.stat().st_size//1024}KB')
